# Task 2 — Image Classification with EfficientNet-B0  
This notebook trains an EfficientNet-B0 model using 5-fold cross-validation  
with mixed precision (AMP) and CUDA acceleration.


In [23]:
import torch
import torch.nn as nn
import numpy as np
from sklearn.model_selection import KFold
from torch.utils.data import DataLoader, Subset
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch import amp
from tqdm import tqdm
import importlib

from models import create_efficientnet_b0
import datasets
importlib.reload(datasets)
from datasets import Task2TrainingDataset224, Task2TestDataset224

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True
device

device(type='cuda')

In [24]:
train_dataset = Task2TrainingDataset224()
test_dataset  = Task2TestDataset224()

n_samples = len(train_dataset)
all_indices = np.arange(n_samples)

print("Total samples:", n_samples)



Total samples: 10000


In [25]:
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

num_epochs = 50
batch_size = 160
val_frequency = 2

fold_results = []


In [30]:
def train_one_fold(train_idx, val_idx):
    train_subset = Subset(train_dataset, train_idx)
    val_subset   = Subset(train_dataset, val_idx)

    num_workers = 8  # try 8; if CPU is still not maxed, try 12

    train_loader = DataLoader(
        train_subset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=4,     # needs num_workers > 0
    )

    val_loader = DataLoader(
        val_subset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=4,
    )

    model = create_efficientnet_b0(num_classes=10).to(device)
    model = model.to(memory_format=torch.channels_last)

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs)

    best_acc = 0.0

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0

        for x, y in train_loader:
            # non-blocking transfers + channels_last
            x = x.to(device, non_blocking=True)
            x = x.to(memory_format=torch.channels_last)
            y = y.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            pred = model(x)
            loss = criterion(pred, y)

            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        scheduler.step()
        running_loss /= len(train_loader)

        if (epoch + 1) % val_frequency == 0 or epoch + 1 == num_epochs:
            model.eval()
            correct = 0
            total = 0
            with torch.no_grad():
                for x, y in val_loader:
                    x = x.to(device, non_blocking=True)
                    x = x.to(memory_format=torch.channels_last)
                    y = y.to(device, non_blocking=True)

                    pred = model(x)
                    _, cls = pred.max(1)
                    total += y.size(0)
                    correct += (cls == y).sum().item()

            acc = correct / total
            best_acc = max(best_acc, acc)

            print(
                f"Epoch {epoch+1}/{num_epochs} | "
                f"Loss={running_loss:.4f} | Val Acc={acc:.4f}"
            )

    return best_acc



In [29]:
print("Starting 5-fold cross validation...\n")

for fold, (train_idx, val_idx) in enumerate(kfold.split(all_indices)):
    print(f"\n===== Fold {fold+1} =====")
    best_acc = train_one_fold(train_idx, val_idx)
    fold_results.append(best_acc)

print("\nResults per fold:", fold_results)
print("Mean accuracy:", np.mean(fold_results))


Starting 5-fold cross validation...


===== Fold 1 =====


TritonMissing: Cannot find a working triton installation. Either the package is not installed or it is too old. More information on installing Triton can be found at: https://github.com/triton-lang/triton

Set TORCHDYNAMO_VERBOSE=1 for the internal stack trace (please do this especially if you're reporting a bug to PyTorch). For even more developer context, set TORCH_LOGS="+dynamo"


In [ ]:
import json

try:
    if len(fold_results) == 0:
        raise ValueError("fold_results is empty")
except NameError:
    raise NameError(
        "fold_results is not defined. "
        "Please run Cells 3, 4, and 5 first to perform cross-validation."
    )

with open("@generated-data/task2_results.json", "w") as f:
    json.dump({
        "fold_results": fold_results,
        "mean_acc": float(np.mean(fold_results))
    }, f, indent=4)

print("Results saved to task2_results.json")


Results saved to task2_results.json


In [ ]:
from datasets import KaggleTestDataset224, INV_LABELS_DICT

# ============================================================
# Final Training on ALL 10 000 Training Images
# ============================================================

batch_size = 160
num_epochs = 50

train_dataset = Task2TrainingDataset224()
kaggle_test_dataset = KaggleTestDataset224()

full_train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

final_model = create_efficientnet_b0(num_classes=10).to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(final_model.parameters(), lr=5e-4, weight_decay=1e-5)
scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs)
scaler = amp.GradScaler("cuda")

print("\n==============================================")
print("Training FINAL model on all 10 000 train images")
print("==============================================\n")

for epoch in range(num_epochs):
    final_model.train()
    running_loss = 0.0

    for x, y in full_train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)

        with amp.autocast("cuda"):
            pred = final_model(x)
            loss = criterion(pred, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()

    scheduler.step()
    print(f"Epoch {epoch+1}/{num_epochs} | Loss = {running_loss:.4f}")

torch.save(final_model.state_dict(), "task2_final_model_all_train.pth")
print("\nFinal model saved to task2_final_model_all_train.pth\n")



Training FINAL model on all 10 000 train images

Epoch 1/50 | Loss = 79.2622
Epoch 2/50 | Loss = 51.4063
Epoch 3/50 | Loss = 47.2274
Epoch 4/50 | Loss = 44.3994
Epoch 5/50 | Loss = 43.4377
Epoch 6/50 | Loss = 42.2640
Epoch 7/50 | Loss = 42.5929
Epoch 8/50 | Loss = 42.4446
Epoch 9/50 | Loss = 42.9243
Epoch 10/50 | Loss = 42.6305
Epoch 11/50 | Loss = 41.7431
Epoch 12/50 | Loss = 41.9646
Epoch 13/50 | Loss = 41.3545
Epoch 14/50 | Loss = 41.2148
Epoch 15/50 | Loss = 40.6983
Epoch 16/50 | Loss = 40.5940
Epoch 17/50 | Loss = 40.5440
Epoch 18/50 | Loss = 40.6535
Epoch 19/50 | Loss = 40.7648
Epoch 20/50 | Loss = 40.8335
Epoch 21/50 | Loss = 40.4895
Epoch 22/50 | Loss = 40.4376
Epoch 23/50 | Loss = 40.9052
Epoch 24/50 | Loss = 40.5112
Epoch 25/50 | Loss = 40.3842
Epoch 26/50 | Loss = 40.4946
Epoch 27/50 | Loss = 40.5231
Epoch 28/50 | Loss = 40.2134
Epoch 29/50 | Loss = 40.0514
Epoch 30/50 | Loss = 40.0883
Epoch 31/50 | Loss = 40.0805
Epoch 32/50 | Loss = 39.9883
Epoch 33/50 | Loss = 40.1407
Ep

In [ ]:
import os
import pandas as pd

# ============================================================
# Inference on the 2 000 Kaggle test images
# ============================================================

batch_size = 160

kaggle_test_dataset = KaggleTestDataset224()

kaggle_test_loader = DataLoader(
    kaggle_test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

model_path = "task2_final_model_all_train.pth"
if not os.path.exists(model_path):
    raise FileNotFoundError(
        f"Model file '{model_path}' not found. "
        "Please run Cell 7 first to train and save the model."
    )

model_test = create_efficientnet_b0(num_classes=10).to(device)
model_test.load_state_dict(torch.load(model_path))
model_test.eval()

all_filenames = []
all_pred_indices = []

with torch.no_grad():
    for x, fnames in kaggle_test_loader:
        x = x.to(device)
        logits = model_test(x)
        _, cls = logits.max(1)
        all_filenames.extend(list(fnames))
        all_pred_indices.extend(cls.cpu().numpy().tolist())

# Map indices back to label strings
all_pred_labels = [INV_LABELS_DICT[idx] for idx in all_pred_indices]

submission_df = pd.DataFrame({
    "id": all_filenames,
    "label": all_pred_labels
})

submission_df = submission_df.sort_values("id")  # optional, but nice
submission_df.to_csv("task2_kaggle_submission.csv", index=False)

print("Saved Kaggle submission file: task2_kaggle_submission.csv")


Saved Kaggle submission file: task2_kaggle_submission.csv
